# Patrón de Comportamiento: Interpreter

## Introducción
El patrón Interpreter define una representación para la gramática de un lenguaje y un intérprete que usa esa representación para interpretar oraciones del lenguaje.

## Objetivos
- Comprender cómo interpretar expresiones de un lenguaje.
- Identificar cuándo es útil el patrón Interpreter.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: Calculadora de expresiones matemáticas**
Una calculadora puede interpretar y evaluar expresiones como '2 + 3 * 4'.

**¿Dónde se usa en proyectos reales?**
En compiladores, motores de reglas, procesamiento de expresiones, etc.

## Sin patrón Interpreter (forma errónea)
Las expresiones se evalúan con código procedural y condicionales.

In [1]:
def evaluar(expr: str) -> float:
    return eval(expr)

print(evaluar('2 + 3 * 4'))

14


## Con patrón Interpreter (forma correcta)
Cada tipo de expresión se representa como una clase.

In [2]:
class Expresion:
    def interpretar(self) -> float:
        pass

class Numero(Expresion):
    def __init__(self, valor: float) -> None:
        self.valor = valor

    def interpretar(self) -> float:
        return self.valor

class Suma(Expresion):
    def __init__(self, izquierda: Expresion, derecha: Expresion) -> None:
        self.izquierda = izquierda
        self.derecha = derecha

    def interpretar(self) -> float:
        return self.izquierda.interpretar() + self.derecha.interpretar()

class Multiplicacion(Expresion):
    def __init__(self, izquierda: Expresion, derecha: Expresion) -> None:
        self.izquierda = izquierda
        self.derecha = derecha

    def interpretar(self) -> float:
        return self.izquierda.interpretar() * self.derecha.interpretar()

expr = Suma(Numero(2), Multiplicacion(Numero(3), Numero(4)))
print(expr.interpretar())

14


## UML del patrón Interpreter
```plantuml
@startuml
interface Expresion {
    + interpretar()
}
class Numero {
    + interpretar()
}
class Suma {
    + interpretar()
}
class Multiplicacion {
    + interpretar()
}
Expresion <|.. Numero
Expresion <|.. Suma
Expresion <|.. Multiplicacion
@enduml
```

## Otro ejemplo de la vida real: Motor de reglas de descuentos
**Contexto:** un sistema de promociones de e-commerce necesita evaluar condiciones de negocio como "el monto es mayor a 100 Y el cliente es frecuente" para decidir si aplica un descuento. Estas reglas cambian con frecuencia (marketing las ajusta constantemente) y se combinan de formas distintas — justo lo que Interpreter permite representar como un árbol de objetos evaluables.

### Sin patrón (forma errónea)
Las reglas de negocio quedan escondidas dentro de condicionales de Python, imposibles de combinar o inspeccionar sin leer el código.

In [3]:
def aplica_descuento(monto: float, es_cliente_frecuente: bool) -> bool:
    if monto > 100 and es_cliente_frecuente:
        return True
    elif monto > 500:
        return True
    return False

# Cada nueva combinación de condiciones obliga a reescribir esta función
print(aplica_descuento(150, True))
print(aplica_descuento(600, False))

True
True


### Con patrón (forma correcta)
Cada condición (`MontoMayorQue`, `EsClienteFrecuente`) y cada operador lógico (`Y`, `O`) es un objeto `Regla` que se puede combinar libremente en un árbol, sin tocar código de Python cada vez que cambia la promoción.

In [4]:
class Regla:
    def evaluar(self, contexto: dict) -> bool:
        raise NotImplementedError

class MontoMayorQue(Regla):
    def __init__(self, umbral: float) -> None:
        self.umbral = umbral

    def evaluar(self, contexto: dict) -> bool:
        return contexto['monto'] > self.umbral

class EsClienteFrecuente(Regla):
    def evaluar(self, contexto: dict) -> bool:
        return contexto.get('es_cliente_frecuente', False)

class Y(Regla):
    def __init__(self, izquierda: Regla, derecha: Regla) -> None:
        self.izquierda = izquierda
        self.derecha = derecha

    def evaluar(self, contexto: dict) -> bool:
        return self.izquierda.evaluar(contexto) and self.derecha.evaluar(contexto)

class O(Regla):
    def __init__(self, izquierda: Regla, derecha: Regla) -> None:
        self.izquierda = izquierda
        self.derecha = derecha

    def evaluar(self, contexto: dict) -> bool:
        return self.izquierda.evaluar(contexto) or self.derecha.evaluar(contexto)


regla_promocion = O(
    Y(MontoMayorQue(100), EsClienteFrecuente()),
    MontoMayorQue(500)
)

print(regla_promocion.evaluar({'monto': 150, 'es_cliente_frecuente': True}))
print(regla_promocion.evaluar({'monto': 600, 'es_cliente_frecuente': False}))
print(regla_promocion.evaluar({'monto': 150, 'es_cliente_frecuente': False}))

True
True
False


### UML del ejemplo de reglas de descuento
```plantuml
@startuml
abstract class Regla {
    + evaluar(contexto)
}
class MontoMayorQue
class EsClienteFrecuente
class Y
class O
Regla <|-- MontoMayorQue
Regla <|-- EsClienteFrecuente
Regla <|-- Y
Regla <|-- O
Y --> Regla
O --> Regla
@enduml
```

### ¿Dónde más se usa Interpreter?
- **Motores de reglas de negocio:** exactamente este ejemplo — sistemas de precios, promociones y aprobación de crédito que combinan condiciones sin tocar código Python.
- **Filtros de búsqueda avanzada:** interpretar consultas como `precio < 100 AND categoria = "ropa"` en un buscador de e-commerce.
- **Lenguajes de configuración:** interpretar expresiones de un archivo de configuración (reglas de firewall, políticas de permisos) representadas como árbol de condiciones.
- **Compiladores e intérpretes de lenguajes:** cada construcción del lenguaje (suma, condicional, bucle) se modela como una clase que sabe "interpretarse" a sí misma.
- **Validadores de esquemas:** interpretar reglas de validación de JSON/formularios (`requerido AND (es_email OR es_telefono)`) como un árbol combinable.

**Ejercicio de reflexión:** agrega una clase `No(Regla)` que invierta el resultado de otra regla (negación lógica). ¿Qué tendrías que cambiar en el resto del sistema para soportarla?

## Actividad
Crea un intérprete para evaluar expresiones lógicas como 'True AND False'.

---
## Explicación de conceptos clave
- **Gramática:** Cada regla se representa como una clase.
- **Extensibilidad:** Se pueden agregar nuevas reglas fácilmente.
- **Aplicación en la vida real:** Útil en compiladores, motores de reglas y procesamiento de expresiones.

## Conclusión
El patrón Interpreter es ideal para procesar lenguajes y expresiones de manera flexible y extensible.